<h2 align='center'>Codebasics ML Course: ML Flow Tutorial</h2>

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

### Experiment 1: Train Logistic Regression Classifier

In [4]:
log_reg = LogisticRegression(C=1, solver='liblinear')
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)
print(classification_report(y_test, y_pred_log_reg))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



### Experiment 2: Train Random Forest Classifier

In [5]:
rf_clf = RandomForestClassifier(n_estimators=30, max_depth=3)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       270
           1       0.95      0.67      0.78        30

    accuracy                           0.96       300
   macro avg       0.96      0.83      0.88       300
weighted avg       0.96      0.96      0.96       300



### Experiment 3: Train XGBoost

In [6]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



### Experiment 4: Handle class imbalance using SMOTETomek and then Train XGBoost

In [7]:
pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [9]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



<h2 align="center" style="color:blue">Track Experiments Using MLFlow</h2>

In [14]:
models = [
    (
        "Logistic Regression", 
        LogisticRegression(C=1, solver='liblinear'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        RandomForestClassifier(n_estimators=30, max_depth=3), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [15]:
reports = []

for model_name, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [17]:
print(reports)

[{'0': {'precision': 0.9834710743801653, 'recall': 0.8814814814814815, 'f1-score': 0.9296875, 'support': 270.0}, '1': {'precision': 0.4482758620689655, 'recall': 0.8666666666666667, 'f1-score': 0.5909090909090909, 'support': 30.0}, 'accuracy': 0.88, 'macro avg': {'precision': 0.7158734682245654, 'recall': 0.8740740740740741, 'f1-score': 0.7602982954545454, 'support': 300.0}, 'weighted avg': {'precision': 0.9299515531490453, 'recall': 0.88, 'f1-score': 0.8958096590909091, 'support': 300.0}}, {'0': {'precision': 0.988, 'recall': 0.9148148148148149, 'f1-score': 0.95, 'support': 270.0}, '1': {'precision': 0.54, 'recall': 0.9, 'f1-score': 0.675, 'support': 30.0}, 'accuracy': 0.9133333333333333, 'macro avg': {'precision': 0.764, 'recall': 0.9074074074074074, 'f1-score': 0.8125, 'support': 300.0}, 'weighted avg': {'precision': 0.9431999999999999, 'recall': 0.9133333333333333, 'f1-score': 0.9225, 'support': 300.0}}, {'0': {'precision': 0.9814126394052045, 'recall': 0.9777777777777777, 'f1-scor

In [20]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow
mlflow.set_tracking_uri("sqlite:///C:/Users/Chandra Mouli/PycharmProjects1/Rough/mlruns.db")

In [22]:
print(models)

[('Logistic Regression', LogisticRegression(C=1, solver='liblinear'), (array([[ 1.18673836,  1.51144074,  0.78490373, ..., -0.61229492,
        -0.13830257, -0.24753395],
       [-1.28810271, -1.03855344, -2.07092052, ...,  0.49607021,
        -1.50376955,  0.62474155],
       [ 1.66393774,  1.55142135,  2.25024183, ..., -0.69955586,
         1.36600648, -0.68290518],
       ...,
       [-0.08827578, -0.62434749,  0.97821051, ...,  0.18885517,
         1.41672379, -0.28437855],
       [ 0.03549707, -0.1123638 ,  0.34255115, ...,  0.02579732,
         0.42877693, -0.10060604],
       [-0.10496962, -0.11927929, -0.09860757, ...,  0.05012454,
        -0.02735928,  0.03041877]]), array([0, 0, 1, ..., 1, 1, 1])), (array([[ 0.12763692,  0.53885434, -0.67754938, ..., -0.17119613,
        -1.04870039,  0.1959492 ],
       [-2.10542071, -1.44658591, -3.8930874 , ...,  0.74058273,
        -3.14736801,  1.16957702],
       [-0.28933147, -0.48475775,  0.04406009, ...,  0.18182712,
         0.35313

In [23]:
# Initialize MLflow
mlflow.set_experiment("Anomaly Detection")
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000/")

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_param("model", model_name)
        mlflow.log_metric('accuracy', report['accuracy'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('f1_score_macro', report['macro avg']['f1-score'])        
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  

2026/05/17 18:19:45 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly Detection' does not exist. Creating a new experiment.
2026/05/17 18:19:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 18:19:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/2/runs/d1df07aa303840598e9de63d909d1f77
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/05/17 18:20:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 18:20:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/2/runs/bd7eb0867b584cec80a75e359fd8c301
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/05/17 18:20:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/2/runs/df2a0c98b020499b83836b14f64f0a36
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/05/17 18:20:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/2/runs/4e02af55d0b14ef589f05f350e5aad64
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [26]:
model_name = "XGBClassifier With SMOTE"
run_id = input("Enter Run Id:")
model_uri = f"runs:/{run_id}/model"
result = mlflow.register_model(
    model_uri,model_name
)

Enter Run Id: 4e02af55d0b14ef589f05f350e5aad64


Registered model 'XGBClassifier With SMOTE' already exists. Creating a new version of this model...
2026/05/17 21:55:20 WARNING mlflow.tracking._model_registry.fluent: Run with id 4e02af55d0b14ef589f05f350e5aad64 has no artifacts at artifact path 'model', registering model based on models:/m-7e4ab3dd032645e99c42d3bce732934c instead
2026/05/17 21:55:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBClassifier With SMOTE, version 1
Created version '1' of model 'XGBClassifier With SMOTE'.
